# BERTopic: Topic Modelling on 10-K MD&A Sentences

**Pipeline**

1. GPU setup
2. Load sentence-level data from `Data Preparation/mda_processed_sample.xlsx` and visualise (head 30 rows)
3. Embed sentences with `all-MiniLM-L6-v2`
4. Fit an **initial BERTopic** model — topic names, intertopic distance map, term score bar chart
5. **Coherence vs K** — sweep `min_cluster_size` to find the optimal number of topics
6. Fit the **final model** with the best K + **KeyBERTInspired** representation
7. **Coherence assessment** and **topic diversity** metrics


## 1. GPU Setup


In [ ]:
import torch

# Detect best available device: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device : {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU count    : {torch.cuda.device_count()}")
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
elif DEVICE == "mps":
    print("Apple M-series GPU detected — PyTorch MPS backend active.")
    print("SentenceTransformer will use MPS automatically for embedding.")
else:
    print("No GPU found — running on CPU (embeddings will be slower).")


Using device : cuda
GPU count    : 1
GPU name     : NVIDIA GeForce RTX 3090
VRAM         : 25.3 GB


In [ ]:
# GPU diagnostics (NVIDIA only)
import subprocess, shutil

if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)


Tue Mar 31 22:49:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
| 41%   28C    P8             21W /  350W |    9440MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Imports & Setup


In [ ]:
# Install BERTopic and dependencies (run once)
!pip install bertopic sentence-transformers umap-learn hdbscan gensim wandb -q

# cuML: GPU-accelerated UMAP + HDBSCAN (RAPIDS - requires CUDA)
# Auto-detects your CUDA version and installs the matching wheel.
import subprocess, sys

try:
    import cuml

    print(f"cuML {cuml.__version__} already installed")
except ImportError:
    import torch

    cuda_ver = torch.version.cuda
    if cuda_ver:
        major = cuda_ver.split(".")[0]
        pkg = f"cuml-cu{major}"
        print(f"Installing {pkg} for CUDA {cuda_ver} ...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                pkg,
                "--extra-index-url",
                "https://pypi.nvidia.com",
                "-q",
            ],
            check=True,
        )
        print("cuML installed - restart kernel if this is the first install.")
    else:
        print("No CUDA detected - cuML skipped; will use CPU UMAP/HDBSCAN.")

cuML 26.02.000 already installed


In [ ]:
import warnings, random, time
import os

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from IPython.display import display

# BERTopic stack
from sentence_transformers import SentenceTransformer
from umap import UMAP as CpuUMAP
from hdbscan import HDBSCAN as CpuHDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator
import wandb

# cuML: GPU-accelerated UMAP + HDBSCAN (graceful fallback to CPU if unavailable)
try:
    from cuml.manifold import UMAP as GpuUMAP
    from cuml.cluster import HDBSCAN as GpuHDBSCAN

    USE_CUML = True
    print("cuML detected - UMAP and HDBSCAN will run on GPU.")
except ImportError:
    GpuUMAP = CpuUMAP
    GpuHDBSCAN = CpuHDBSCAN
    USE_CUML = False
    print("cuML not available - falling back to CPU UMAP / HDBSCAN.")

# Coherence scoring
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

plt.rcParams["figure.dpi"] = 110
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


class WandbLogger:
    """Minimal helper for hyperparameters and coherence logging."""

    def __init__(self, project, run_name=None, api_key=None):
        if api_key:
            wandb.login(key=api_key, relogin=True)
        self.run = wandb.init(project=project, name=run_name)

    def log_hparams(self, params):
        self.run.config.update(params, allow_val_change=True)

    def log_metrics(self, metrics):
        wandb.log(metrics)

    def log_coherence(self, stage, score, elapsed_sec=None, **extra):
        payload = {f"coherence/{stage}": float(score)}
        if elapsed_sec is not None:
            payload[f"coherence/{stage}_time_sec"] = float(elapsed_sec)
        for k, v in extra.items():
            payload[f"coherence/{stage}_{k}"] = v
        wandb.log(payload)

    def finish(self):
        wandb.finish()


WB_PROJECT = "text-mining"
WB_RUN_NAME = "bertopic-run"
WB_API_KEY = os.getenv(
    "WANDB_API_KEY", ""
)  # Preferred: set env var instead of hardcoding.

wb = WandbLogger(
    project=WB_PROJECT,
    run_name=WB_RUN_NAME,
    api_key=WB_API_KEY if WB_API_KEY else None,
)

print("All imports OK")
print("W&B logger ready at notebook start.")

cuML detected - UMAP and HDBSCAN will run on GPU.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /common/home/users/w/wlcheng.2023/.netrc.
wandb: Currently logged in as: wlunlun1212 (wlunlun1212-singapore-management-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


All imports OK
W&B logger ready at notebook start.


In [ ]:
# ==================================================================
# RUN 2 HYPERPARAMETERS  (fixed - no sweep needed)
# ==================================================================
MIN_CLUSTER_SIZE = 150  # HDBSCAN - min sentences per topic
MIN_SAMPLES = 15  # HDBSCAN - controls cluster density
MIN_DF = 5  # CountVectorizer - min doc-frequency
NGRAM_RANGE = (1, 3)  # CountVectorizer - up to trigrams
BEST_MCS = MIN_CLUSTER_SIZE  # used by final model directly

print(f"Config  min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}")
print(f"        min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")

wb.log_hparams(
    {
        "min_cluster_size": MIN_CLUSTER_SIZE,
        "min_samples": MIN_SAMPLES,
        "min_df": MIN_DF,
        "ngram_low": NGRAM_RANGE[0],
        "ngram_high": NGRAM_RANGE[1],
        "seed": SEED,
        "device": DEVICE,
        "use_cuml": USE_CUML,
    }
)
print("W&B hyperparameters logged.")

Config  min_cluster_size=150  min_samples=15
        min_df=5  ngram_range=(1, 3)
W&B hyperparameters logged.


## 3. Data Loading & Preparation

Load `mda_processed_sample.xlsx` (wide format: 1 row per filing, columns `sent_1…sent_N`)
and melt into a sentence-level long DataFrame used throughout the notebook.


In [ ]:
import ast
import json
from pathlib import Path


def find_data_path() -> str:
    """Find datasets/final/mda_shared_processed.csv from common notebook CWDs."""
    rel_target = Path("datasets/final/mda_shared_processed.csv")
    cwd = Path.cwd().resolve()

    # 1) Try current directory and its parents (handles most VS Code notebook CWDs).
    for base in [cwd, *cwd.parents]:
        candidate = (base / rel_target).resolve()
        if candidate.exists():
            return str(candidate)

    # 2) Try likely project roots even if kernel CWD is elsewhere.
    project_root_hints = [
        Path("/Users/lunlun/Downloads/Github/textmining_grp6"),
        Path.home() / "Downloads" / "Github" / "textmining_grp6",
        Path.home() / "textmining_grp6",
    ]
    for root in project_root_hints:
        candidate = (root / rel_target).resolve()
        if candidate.exists():
            return str(candidate)

    # 3) Legacy explicit relative paths used previously.
    for p in (
        Path("../datasets/final/mda_shared_processed.csv"),
        Path("datasets/final/mda_shared_processed.csv"),
    ):
        if p.exists():
            return str(p.resolve())

    raise FileNotFoundError(
        "Could not find mda_shared_processed.csv. "
        "Expected under datasets/final/ from the notebook CWD, project root hints, or parent directories."
    )


DATA_PATH = find_data_path()
print(f"Using data file: {DATA_PATH}")


def _parse_sentences(value):
    """Parse a JSON-like list safely; return [] when malformed."""
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []

    text = value.strip()
    if not text:
        return []

    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


def load_mda_sentences(path: str) -> pd.DataFrame:
    """Load CSV and explode sentence lists into one row per sentence."""
    raw = pd.read_csv(path, usecols=["doc_id", "sentences"])
    raw["sentences"] = raw["sentences"].apply(_parse_sentences)

    rows = []
    for _, r in raw.iterrows():
        for idx, sent in enumerate(r["sentences"]):
            sent = str(sent).strip()
            if sent:
                rows.append(
                    {"doc": r["doc_id"], "sent_idx": idx, "processed_text": sent}
                )

    return pd.DataFrame(rows, columns=["doc", "sent_idx", "processed_text"])


df = load_mda_sentences(DATA_PATH)
docs = df["processed_text"].tolist()

print(f"Documents   : {df['doc'].nunique()}")
print(f"Sentences   : {len(df):,}")
print(f"Avg sent/doc: {len(df) / df['doc'].nunique():.1f}")

Using data file: /common/home/users/w/wlcheng.2023/textmining_grp6/datasets/final/mda_shared_processed.csv
Documents   : 18025
Sentences   : 1,568,758
Avg sent/doc: 87.0


In [ ]:
df.drop_duplicates(subset=["sentence"], inplace=True)
print(f"Duplicates dropped: {len(df_raw) - len(df):,}")
print(f"Final row count: {len(df):,}")

In [18]:
# Convert sentence-level rows to chunk-level rows (5 sentences per chunk)
CHUNK_SIZE = 5

if "sent_idx" in df.columns:
    sentence_df = df.copy()
elif "df_sent" in globals() and "sent_idx" in df_sent.columns:
    sentence_df = df_sent.copy()
else:
    raise ValueError("Need sentence-level data with sent_idx. Run Cell 10 first.")

sentence_df = sentence_df.sort_values(["doc", "sent_idx"], kind="mergesort")
sentence_df["chunk_idx"] = sentence_df["sent_idx"] // CHUNK_SIZE

df = sentence_df.groupby(["doc", "chunk_idx"], as_index=False).agg(
    processed_text=("processed_text", lambda s: " ".join(s.astype(str)))
)

df["processed_text"] = (
    df["processed_text"].str.replace(r"\s+", " ", regex=True).str.strip()
)
docs = df["processed_text"].tolist()

print(f"=== First 30 chunks ({CHUNK_SIZE} sentences per row) ===")
print(f"Chunk rows: {len(df):,}")
display(df.head(30))

=== First 30 chunks (5 sentences per row) ===
Chunk rows: 318,580


,doc,chunk_idx,processed_text
0,1-800-PetMeds_10-K_2020-05-26,0,the companys common stock is traded on the nas...
1,1-800-PetMeds_10-K_2020-05-26,1,on an ongoing basis we re-evaluate our judgmen...
2,1-800-PetMeds_10-K_2020-05-26,2,the allowance for doubtful accounts was approx...
3,1-800-PetMeds_10-K_2020-05-26,3,due to covid consumer demand has increased for...
4,1-800-PetMeds_10-K_2020-05-26,4,for the quarters ended june september december...
5,1-800-PetMeds_10-K_2020-05-26,5,going forward gross profit may be adversely af...
6,1-800-PetMeds_10-K_2020-05-26,6,historically the advertising environment fluct...
7,1-800-PetMeds_10-K_2020-05-26,7,th can be attributed to an increase in new pro...
8,1-800-PetMeds_10-K_2020-05-26,8,the company estimates its effective tax rate w...
9,1-800-PetMeds_10-K_2020-05-26,9,for the quarters ended june september december...


## 4. Sentence Embeddings


In [19]:
EMBED_MODEL = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL} on device={DEVICE} ...")
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

# RTX 3090 (24 GB VRAM) handles 1024 comfortably; MPS 256; CPU 64
BATCH_SIZE = 1024 if DEVICE == "cuda" else (256 if DEVICE == "mps" else 64)
print(f"Batch size : {BATCH_SIZE}  (device={DEVICE})")

_t0 = time.perf_counter()
embeddings = embed_model.encode(
    docs,
    show_progress_bar=True,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,  # unit-length → cosine ≡ dot, slightly faster UMAP
)
_embed_sec = time.perf_counter() - _t0
print(f"Embeddings shape : {embeddings.shape}")
print(f"Embedding time   : {_embed_sec:.1f}s")

Loading embedding model: all-MiniLM-L6-v2 on device=cuda ...
Batch size : 1024  (device=cuda)


Batches:   0%|          | 0/312 [00:00<?, ?it/s]

Embeddings shape : (318580, 384)
Embedding time   : 204.6s


In [20]:
# Pre-compute UMAP reduction ONCE — reused by every model fit and the sweep.
# cuML UMAP (GPU): ~10-20s on RTX 3090, no PCA init needed.
# CPU UMAP fallback: PCA warm-start for 3-5x faster convergence.

_t1 = time.perf_counter()

if USE_CUML:
    print("Fitting UMAP on GPU (cuML) ...")
    _umap_base = GpuUMAP(
        n_components=5,
        n_neighbors=15,
        min_dist=0.0,
        metric="cosine",
        random_state=SEED,
    )
else:
    print("Fitting UMAP on CPU (PCA-initialised) ...")

    def _rescale(x):
        x = np.array(x, copy=True).astype(np.float32)
        x /= np.std(x[:, 0]) * 10_000
        return x

    _pca_init = _rescale(
        PCA(n_components=5, random_state=SEED).fit_transform(embeddings)
    )
    _umap_base = CpuUMAP(
        n_components=10,
        n_neighbors=50,
        min_dist=0.0,
        metric="cosine",
        init=_pca_init,
        low_memory=False,
        random_state=SEED,
    )

umap_embeddings = _umap_base.fit_transform(embeddings)
_umap_sec = time.perf_counter() - _t1
backend = "GPU cuML" if USE_CUML else "CPU PCA-init"
print(f"UMAP shape : {umap_embeddings.shape}")
print(f"UMAP time  : {_umap_sec:.1f}s  ({backend})")


class PassthroughUMAP(BaseEstimator):
    """Returns pre-computed UMAP embeddings — skips re-fitting for all downstream calls."""

    def __init__(self, precomputed):
        self.precomputed = precomputed

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.precomputed

    def fit_transform(self, X, y=None):
        return self.precomputed


print("PassthroughUMAP ready.")

Fitting UMAP on GPU (cuML) ...
UMAP shape : (318580, 5)
UMAP time  : 8.0s  (GPU cuML)
PassthroughUMAP ready.


## 5. Initial BERTopic Model


In [21]:
umap_model = PassthroughUMAP(umap_embeddings)

if USE_CUML:
    hdbscan_model = GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_model = CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t3 = time.perf_counter()
topics, probs = topic_model.fit_transform(docs, embeddings)
_initial_sec = time.perf_counter() - _t3
print(f"Initial model fit time: {_initial_sec:.1f}s")

n_topics = len(set(t for t in topics if t != -1))
n_outliers = sum(1 for t in topics if t == -1)
print(f"\nTopics found : {n_topics}")
print(f"Outliers     : {n_outliers:,} ({n_outliers / len(docs):.1%})")

wb.log_metrics(
    {
        "initial_fit_time_sec": float(_initial_sec),
        "initial_topics_found": int(n_topics),
        "initial_outliers": int(n_outliers),
        "initial_outlier_ratio": float(n_outliers / len(docs)),
    }
)
print("W&B metrics logged for initial model fit.")

2026-03-31 22:54:04,143 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-31 22:54:04,144 - BERTopic - Dimensionality - Completed ✓
2026-03-31 22:54:04,148 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-31 22:54:08,338 - BERTopic - Cluster - Completed ✓
2026-03-31 22:54:08,388 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-31 22:54:57,410 - BERTopic - Representation - Completed ✓


Initial model fit time: 64.0s

Topics found : 278
Outliers     : 163,213 (51.2%)
W&B metrics logged for initial model fit.


In [22]:
# -- Coherence helper (Gensim Cv) - used throughout the notebook ----------------
import os

_N_WORKERS = max(1, os.cpu_count() - 1)

# Pre-build tokenised corpus + dictionary once - reused by every coherence call
_tokenised = [d.lower().split() for d in docs]
_dictionary = Dictionary(_tokenised)


def compute_coherence(model, stage=None, extra_log=None):
    """Compute Cv coherence and optionally log it to W&B in the same call."""
    t0 = time.perf_counter()
    topic_words = [
        [w for w, _ in words if w]
        for tid, words in model.get_topics().items()
        if tid != -1
    ]

    if not topic_words:
        score = 0.0
    else:
        try:
            cm = CoherenceModel(
                topics=topic_words,
                texts=_tokenised,
                dictionary=_dictionary,
                coherence="c_v",
                processes=_N_WORKERS,
            )
            score = float(cm.get_coherence())
        except Exception:
            score = 0.0

    elapsed = time.perf_counter() - t0

    if stage is not None and "wb" in globals():
        payload = extra_log.copy() if isinstance(extra_log, dict) else {}
        wb.log_coherence(
            stage=stage,
            score=score,
            elapsed_sec=elapsed,
            **payload,
        )

    return score, elapsed

In [23]:
# ── Topic Names via get_topic_info() ─────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print("=== Topic Overview ===")
display(topic_info)

print("\n=== Topic Names ===")
for _, row in topic_info.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

=== Topic Overview ===


,Topic,Count,Name,Representation,Representative_Docs
0,-1,163213,-1_cash equivalents_cash cash_equivalents_cash...,"[cash equivalents, cash cash, equivalents, cas...",[we are currently in the process of evaluating...
1,0,9254,0_general administrative_sales marketing expen...,"[general administrative, sales marketing expen...",[we expect that our sales and marketing expens...
2,1,4585,1_reporting unit_value reporting_fair value re...,"[reporting unit, value reporting, fair value r...",[we currently have no intangible assets with i...
3,2,4514,2_asu_fasb issued_fasb_guidance,"[asu, fasb issued, fasb, guidance, topic, adop...",[asu applies to all entities that present a cl...
4,3,4219,3_privacy_gdpr_data protection_personal data,"[privacy, gdpr, data protection, personal data...",[we cannot fully predict how the data protecti...
...,...,...,...,...,...
274,273,153,273_mosfets_cost reduction_cost reduction prog...,"[mosfets, cost reduction, cost reduction progr...",[our variable cost reduction efforts include i...
275,274,152,274_fcc_neutrality_fcc rules regulations_opera...,"[fcc, neutrality, fcc rules regulations, opera...",[as a provider of communications products we a...
276,275,151,275_vehicle programs_fixedincome_vehicle_asset...,"[vehicle programs, fixedincome, vehicle, asset...",[for information regar and to our consolidated...
277,276,150,276_wu_license agreement_breach license_milestone,"[wu, license agreement, breach license, milest...",[royalties payable to wu under the wu license ...



=== Topic Names ===
  Topic   0: 0_general administrative_sales marketing expenses_sales marketing_expense net
  Topic   1: 1_reporting unit_value reporting_fair value reporting_reporting units
  Topic   2: 2_asu_fasb issued_fasb_guidance
  Topic   3: 3_privacy_gdpr_data protection_personal data
  Topic   4: 4_class common stock_class common_class_voting
  Topic   5: 5_repurchase_repurchase program_stock repurchase_repurchases
  Topic   6: 6_intellectual property rights_property rights_intellectual property_intellectual
  Topic   7: 7_awards_grant_grant date_vesting
  Topic   8: 8_vsoe_elements_performance obligation_standalone
  Topic   9: 9_bitcoin_mining_miners_digital assets
  Topic  10: 10_nongaap_financial measures_gaap_nongaap financial
  Topic  11: 11_term loan_credit agreement_revolving_revolving credit
  Topic  12: 12_tax assets_deferred tax assets_deferred tax_valuation allowance
  Topic  13: 13_net cash_operating assets liabilities_operating assets_net cash provided
  Topi

## 6. Visualisations — Initial Model

### Intertopic Distance Map

Each bubble represents one topic; bubble **size** is proportional to the number of
documents assigned; **distance** approximates semantic difference. Colours distinguish topics.

### Term Score Bar Chart

Bar height = c-TF-IDF score — how distinctive a term is to its topic vs. all others.


In [ ]:
# ── Intertopic Distance Map (with topic colours) ───────────────────────────────
fig_idm = topic_model.visualize_topics()
fig_idm.update_layout(
    title_text="Intertopic Distance Map — Initial Model",
    height=600,
)
fig_idm.show()

In [ ]:
# ── Term Score Bar Charts ─────────────────────────────────────────────────────
n_show = min(n_topics, 9)  # cap at 9 for readability
fig_bc = topic_model.visualize_barchart(top_n_topics=n_show, n_words=8)
fig_bc.update_layout(
    title_text="Top Terms per Topic — c-TF-IDF Scores (Initial Model)",
    height=700,
)
fig_bc.show()

## 7. Coherence Validation (Run 2 Params — No Sweep)

Hyperparameters are fixed to **Run 2** values (`min_cluster_size=150`, `min_samples=15`,
`min_df=5`, `ngram_range=(1,3)`). We compute a single coherence score to validate
the configuration rather than sweeping across many values.

To re-enable the full sweep, change the config cell above and replace this section.


In [ ]:
# Single coherence check for Run 2 params - no sweep loop needed.
cv_run2, _cv_sec = compute_coherence(
    topic_model,
    stage="run2",
    extra_log={
        "topics": int(n_topics),
        "min_cluster_size": int(MIN_CLUSTER_SIZE),
        "min_samples": int(MIN_SAMPLES),
    },
)

k_results = [{"mcs": MIN_CLUSTER_SIZE, "k": n_topics, "cv": cv_run2}]
best_k_row = k_results[0]

print(
    f"{'min_cluster_size':>18} {'min_samples':>12} {'k (topics)':>12} {'Coherence Cv':>14}"
)
print("-" * 62)
print(f"{MIN_CLUSTER_SIZE:>18} {MIN_SAMPLES:>12} {n_topics:>12} {cv_run2:>14.4f}")
print(f"\nCoherence computed in {_cv_sec:.1f}s")
print("W&B coherence logged: run2")

In [ ]:
# Simple coherence summary bar for the single Run-2 configuration
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(["Run 2"], [cv_run2], color="#2980b9", height=0.4)
ax.axvline(0.55, color="#27ae60", linestyle="--", lw=1.2, label="Good (0.55)")
ax.axvline(0.70, color="#e74c3c", linestyle="--", lw=1.2, label="Very Good (0.70)")
ax.set_xlim(0, max(1.0, cv_run2 + 0.1))
ax.set_xlabel("Coherence Cv", fontsize=11)
ax.set_title(
    f"Run 2 — Cv={cv_run2:.4f}  |  k={n_topics} topics", fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 8. Final BERTopic Model with KeyBERT Representation

We fit the final model using the optimal `min_cluster_size` found above and add
**KeyBERTInspired** as the representation model to produce more semantically precise,
human-readable topic labels than raw c-TF-IDF alone.


In [ ]:
print(f"Final model — min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}")
print(f"             min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")
print("Representation: KeyBERTInspired")
print("-" * 50)

if USE_CUML:
    hdbscan_final = GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_final = CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )

vectorizer_final = CountVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf_final = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

topic_model_final = BERTopic(
    embedding_model=embed_model,
    umap_model=PassthroughUMAP(umap_embeddings),
    hdbscan_model=hdbscan_final,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    representation_model=representation_model,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t4 = time.perf_counter()
topics_final, probs_final = topic_model_final.fit_transform(docs, embeddings)
_final_sec = time.perf_counter() - _t4
print(f"Final model fit time: {_final_sec:.1f}s")

n_topics_final = len(set(t for t in topics_final if t != -1))
n_outliers_final = sum(1 for t in topics_final if t == -1)
print(f"\nFinal Topics  : {n_topics_final}")
print(f"Final Outliers: {n_outliers_final:,} ({n_outliers_final / len(docs):.1%})")

In [ ]:
# ── Final topic names via get_topic_info() → Name column ─────────────────────
topic_info_final = topic_model_final.get_topic_info()
print("=== Final Topic Overview (KeyBERT-enhanced) ===")
display(topic_info_final)

print("\n=== Final Topic Names ===")
for _, row in topic_info_final.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

In [ ]:
# ── Final Intertopic Distance Map (with topic colours) ────────────────────────
fig_final_idm = topic_model_final.visualize_topics()
fig_final_idm.update_layout(
    title_text="Final Model — Intertopic Distance Map",
    height=600,
)
fig_final_idm.show()

In [ ]:
# ── Final Term Score Bar Charts ───────────────────────────────────────────────
n_show_final = min(n_topics_final, 9)
fig_final_bc = topic_model_final.visualize_barchart(
    top_n_topics=n_show_final, n_words=8
)
fig_final_bc.update_layout(
    title_text="Final Model — Top Terms per Topic (KeyBERT Representation)",
    height=700,
)
fig_final_bc.show()

## 9. Coherence Assessment

**Cv coherence** measures how semantically similar the top words within each topic are —
higher is better.

| Cv range  | Interpretation |
| --------- | -------------- |
| < 0.45    | Poor           |
| 0.45–0.55 | Marginal       |
| 0.55–0.70 | Good           |
| > 0.70    | Very good      |


In [ ]:
cv_initial, cv_initial_sec = compute_coherence(
    topic_model,
    stage="initial",
    extra_log={"topics": int(n_topics)},
)
cv_final, cv_final_sec = compute_coherence(
    topic_model_final,
    stage="final",
    extra_log={"topics": int(n_topics_final)},
)

wb.log_metrics(
    {
        "coherence/improvement": float(cv_final - cv_initial),
        "coherence/initial_time_sec": float(cv_initial_sec),
        "coherence/final_time_sec": float(cv_final_sec),
    }
)

print("=" * 55)
print("COHERENCE ASSESSMENT")
print("=" * 55)
print(f"\n  Initial model Cv : {cv_initial:.4f}")
print(f"  Final model Cv   : {cv_final:.4f}")
print(f"  Improvement      : {cv_final - cv_initial:+.4f}")
print("W&B coherence logged: initial, final")

print()
for score, label in [(cv_initial, "Initial model"), (cv_final, "Final model")]:
    if score >= 0.70:
        msg = "VERY GOOD (>=0.70)"
    elif score >= 0.55:
        msg = "GOOD (0.55-0.70)"
    elif score >= 0.45:
        msg = "MARGINAL (0.45-0.55) - consider tuning"
    else:
        msg = "POOR (<0.45)"
    print(f"  {label:<16}: Cv = {score:.4f}  =>  {msg}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    ["Initial", "Final"],
    [cv_initial, cv_final],
    color=["#7fb3d3", "#2980b9"],
    edgecolor="white",
    width=0.4,
)
for bar, val in zip(bars, [cv_initial, cv_final]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.003,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_ylabel("Coherence Cv")
ax.set_title("Initial vs Final Model - Coherence", fontweight="bold")
ax.set_ylim(0, max(cv_initial, cv_final) * 1.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Topic Diversity

**Topic Diversity** = unique words across all topics ÷ total word slots across all topics.

A high score (close to 1.0) means topics use largely distinct vocabulary — good separation.
A low score means topics share many words — possible redundancy or over-splitting.


In [ ]:
# ── Topic Diversity for final model ──────────────────────────────────────────
all_topic_words = []
for tid, words in topic_model_final.get_topics().items():
    if tid != -1:
        all_topic_words.extend([w for w, _ in words if w])

unique_words = set(all_topic_words)
topic_diversity = len(unique_words) / len(all_topic_words) if all_topic_words else 0.0

print("=" * 55)
print("TOPIC DIVERSITY — Final Model")
print("=" * 55)
print(f"\n  Total word slots : {len(all_topic_words)}")
print(f"  Unique words     : {len(unique_words)}")
print(f"  Topic Diversity  : {topic_diversity:.4f}  ({topic_diversity:.1%})")

if topic_diversity >= 0.70:
    div_msg = "HIGH — topics are well-differentiated"
elif topic_diversity >= 0.50:
    div_msg = "MODERATE — some vocabulary overlap between topics"
else:
    div_msg = "LOW — topics share much vocabulary; consider increasing k"
print(f"\n  Interpretation: {div_msg}")

# Visualise word frequency across topics
from collections import Counter

word_freq = Counter(all_topic_words)
top_shared = [(w, c) for w, c in word_freq.most_common(20) if c > 1]

if top_shared:
    words_s, counts_s = zip(*top_shared)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    colors = ["#e74c3c" if c >= 3 else "#e67e22" for c in counts_s]
    ax.barh(words_s, counts_s, color=colors, edgecolor="white")
    ax.invert_yaxis()
    ax.set_xlabel("Number of topics containing this word")
    ax.set_title(
        f"Words Shared Across Topics (Diversity = {topic_diversity:.1%})",
        fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n(Red bars = word appears in 3+ topics — high overlap indicator)")